In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from datasets import load_dataset, DatasetDict
from transformers import pipeline, AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

ds = load_dataset("mteb/imdb")

# ds = DatasetDict({
#     "train": ds["train"].shuffle(seed=42).select(range(500)),
#     "test": ds["test"].shuffle(seed=42).select(range(200)),
# })



def setup_config(rank, target_modules):
    config = LoraConfig(
        r=rank,
        lora_alpha=16,
        target_modules = target_modules,
        lora_dropout=0.1,
        bias='none',
        task_type="SEQ_CLS",
        modules_to_save=['classifier']
    )
    return config

In [3]:
from transformers import TrainingArguments, Trainer
import time
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
import os


import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# config1 = setup_config(8, ["query", "value"])
# config2 = setup_config(16, ["query", "value"])
# config3 = setup_config(16, ["query", "value", 'fc1', 'fc2'])

# # lst_configs = [config1, config2, config3]

# configs_to_test = {
#     "lora_r8_qv": config1,
#     "lora_r16_qv": config2,
#     "lora_r16_qv_fc": config3
# }

def get_target_modules(model_id):
    model_id = model_id.lower()
    if "roberta" in model_id:
        return ["query", "value"]
    elif "deberta" in model_id:
        return ["query_proj", "value_proj"]
    elif "distilbert" in model_id:
        return ["q_lin", "v_lin"]
    else:
        # Default fallback for most BERT-like models
        return ["query", "value"]


def prepare_datasets(model_name, ds):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)
    
    tokenized_dataset = ds.map(tokenize_function, batched=True)
    
    cols_to_remove = [col for col in tokenized_dataset["train"].column_names if col not in ["input_ids", "attention_mask", "label"]]
    tokenized_dataset = tokenized_dataset.remove_columns(cols_to_remove)
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
    tokenized_dataset.set_format("torch")
    
    train_valid = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=42)
    
    train_ds = train_valid["train"]
    val_ds = train_valid["test"]
    test_ds = tokenized_dataset["test"] 

    return train_ds, val_ds, test_ds

# def load_model_and_dataset(model_name, ds):
#     model = AutoModelForSequenceClassification.from_pretrained(
#         model_name,
#         num_labels=2
#     )

#     tokenizer = AutoTokenizer.from_pretrained(model_name)
    
#     tokenized_dataset = ds.map(tokenize_function, batched=True)
#     tokenized_dataset = tokenized_dataset.remove_columns(["text"])
#     tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
#     tokenized_dataset.set_format("torch")
#     train_valid = tokenized_dataset["train"].train_test_split(
#         test_size=0.1,
#         seed=42
#     )
    
    
#     train_ds = train_valid["train"]
#     val_ds = train_valid["test"]
#     test_ds = dataset["test"]

#     return model, train_ds, val_ds, test_d
logs = []
def training_pipeline_lora(model_id, config, run_name,seed, train_ds, val_ds, test_ds):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        use_safetensors=True
    )
    set_seed(seed)

    print(f"--- Training Run: {run_name} ---")
    model = get_peft_model(base_model, config)
    model.print_trainable_parameters()
    
    batch_size = 16
    learning_rate = 2e-5

    safe_model_name = model_id.split("/")[-1]
    out_dir = f"./results/{safe_model_name}_{run_name}_seed{seed}"
    args = TrainingArguments(
        output_dir=out_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=batch_size,
        bf16=True,
        num_train_epochs=2,
        logging_steps=10,
        load_best_model_at_end=True,
        label_names=["labels"],
    )

    trainer = Trainer(
        model,
        args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
    )
    os.system("nvidia-smi")
    start = time.time()
    trainer.train()
    os.system("nvidia-smi")
    end = time.time()
    train_time = end - start
    if torch.cuda.is_available():
        peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        peak_vram = 0
    predictions = trainer.predict(test_ds)

    preds = np.argmax(predictions.predictions, axis=1)
    
    labels = predictions.label_ids
    
    acc = accuracy_score(labels, preds)
        
    f1 = f1_score(labels, preds, average="macro")

    print(f"[{run_name}] Accuracy: {acc:.4f}")
    print(f"[{run_name}] F1 Score: {f1:.4f}")
    print(f"[{run_name}] Training Time: {train_time:.2f} seconds\n")

    logs.append({
        "seed": seed,
        "rank": config.r,
        "lr": learning_rate,
        "run_name": run_name,
        "target": str(config.target_modules),
        "accuracy": acc,
        "f1": f1,
        "train_time": train_time,
        "vram": peak_vram
    })


    del model, trainer, base_model
    torch.cuda.empty_cache()


In [4]:
experiments = [
    {
        "id": "EXP01",
        "model_id": "FacebookAI/roberta-base",
        "r": 16,
        "targets": ["query", "value", "intermediate.dense", "output.dense"], 
        "name": "roberta_r8_qv_ffn"
    },
    # {
    #     "id": "EXP02",
    #     "model_id": "microsoft/deberta-v3-base",
    #     "r": 16,
    #     "targets": ["query_proj", "value_proj", "intermediate.dense", "output.dense"],
    #     "name": "deberta_r16_all"
    # },
    # {
    #     "id": "EXP03",
    #     "model_id": "distilbert/distilbert-base-uncased",
    #     "r": 16,
    #     "targets": ["q_lin", "v_lin", "ffn.lin1", "ffn.lin2"],
    #     "name": "distil_r16_ffn"
    # }
]



for exp in experiments:
    model_id = exp['model_id']
    run_name = exp['name']
    config = setup_config(16, exp['targets'])
    train_ds, val_ds, test_ds = prepare_datasets(model_id, ds)
    for seed in [42, 43]:
        training_pipeline_lora(model_id, config, run_name, seed, train_ds, val_ds, test_ds)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Run: roberta_r8_qv_ffn ---
trainable params: 2,951,426 || all params: 127,598,596 || trainable%: 2.3131


`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,1.000270,0.253885
2,0.854847,0.234214


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Run: roberta_r8_qv_ffn ---
trainable params: 2,951,426 || all params: 127,598,596 || trainable%: 2.3131


Epoch,Training Loss,Validation Loss
1,0.959501,0.247766
2,0.903357,0.225595


[roberta_r8_qv_ffn] Accuracy: 0.9201
[roberta_r8_qv_ffn] F1 Score: 0.9201
[roberta_r8_qv_ffn] Training Time: 559.51 seconds



In [5]:
df = pd.DataFrame(logs)
df.to_csv("exp07_lora_experiment_results.csv", index=False)

In [6]:
from peft import IA3Config, get_peft_model


def setup_config(target_modules, feedforward_modules=None):
    config = IA3Config(
        target_modules = target_modules,
        task_type="SEQ_CLS",
        feedforward_modules=feedforward_modules if feedforward_modules is not None else [],
    )
    return config
    


In [7]:
logs = []
def training_pipeline_adapter(model_id, config, run_name,seed, train_ds, val_ds, test_ds):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        use_safetensors=True
    )
    set_seed(seed)

    print(f"--- Training Run: {run_name} ---")
    model = get_peft_model(base_model, config)
    model.print_trainable_parameters()
    
    batch_size = 16
    learning_rate = 2e-5

    safe_model_name = model_id.split("/")[-1]
    out_dir = f"./results/{safe_model_name}_{run_name}_seed{seed}"
    args = TrainingArguments(
        output_dir=out_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=batch_size,
        bf16=True,
        num_train_epochs=2,
        logging_steps=10,
        load_best_model_at_end=True,
        label_names=["labels"],
    )

    trainer = Trainer(
        model,
        args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
    )
    
    os.system("nvidia-smi")
    start = time.time()
    trainer.train()
    os.system("nvidia-smi")
    end = time.time()
    train_time = end - start
    if torch.cuda.is_available():
        peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        peak_vram = 0
    predictions = trainer.predict(test_ds)

    preds = np.argmax(predictions.predictions, axis=1)
    
    labels = predictions.label_ids
    
    acc = accuracy_score(labels, preds)
        
    f1 = f1_score(labels, preds, average="macro")

    print(f"[{run_name}] Accuracy: {acc:.4f}")
    print(f"[{run_name}] F1 Score: {f1:.4f}")
    print(f"[{run_name}] Training Time: {train_time:.2f} seconds\n")

    logs.append({
        "seed": seed,
        "lr": learning_rate,
        "run_name": run_name,
        "target": str(config.target_modules),
        "ffn": str(config.feedforward_modules),
        "accuracy": acc,
        "f1": f1,
        "train_time": train_time,
        "vram": peak_vram
    })


    del model, trainer, base_model
    torch.cuda.empty_cache()


In [8]:
experiments = [
    # {
    #     "id": "EXP01",
    #     "model_id": "FacebookAI/roberta-base",
    #     "attention": ["value"],
    #     "feedforward": ["intermediate.dense", "output.dense"],
    #     "name": "roberta_ffn"
    # },
    # {
    #     "id": "EXP02",
    #     "model_id": "microsoft/deberta-v3-base",
    #     "attention": ["value_proj"],
    #     "feedforward": ["intermediate.dense", "output.dense"],
    #     "name": "deberta_ffn"
    # },
    {
        "id": "EXP03",
        "model_id": "distilbert/distilbert-base-uncased",
        "attention": ["v_lin"],
        "feedforward": ["ffn.lin1", "ffn.lin2"],
        "name": "distil_ffn"
    }
]



for exp in experiments:
    model_id = exp['model_id']
    run_name = exp['name']
    all_targets = exp['attention'] + exp['feedforward']
    config = setup_config(all_targets, exp['feedforward'])
    train_ds, val_ds, test_ds = prepare_datasets(model_id, ds)
    for seed in [42, 43]:
        training_pipeline_adapter(model_id, config, run_name, seed, train_ds, val_ds, test_ds)




Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Run: distil_ffn ---
trainable params: 619,778 || all params: 67,574,788 || trainable%: 0.9172


Epoch,Training Loss,Validation Loss
1,2.300684,0.561801
2,2.080490,0.514356


[distil_ffn] Accuracy: 0.8066
[distil_ffn] F1 Score: 0.8066
[distil_ffn] Training Time: 220.82 seconds



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Run: distil_ffn ---
trainable params: 619,778 || all params: 67,574,788 || trainable%: 0.9172


Epoch,Training Loss,Validation Loss
1,2.300708,0.561797
2,2.079865,0.514388


[distil_ffn] Accuracy: 0.8068
[distil_ffn] F1 Score: 0.8068
[distil_ffn] Training Time: 222.05 seconds



In [9]:
df = pd.DataFrame(logs)
df.to_csv("exp07_ia3_adapter_experiment_results.csv", index=False)

In [10]:
from transformers import BitsAndBytesConfig

def setup_config(rank, target_modules):
    config = LoraConfig(
        r=rank,
        lora_alpha=16,
        target_modules = target_modules,
        lora_dropout=0.1,
        bias='none',
        task_type="SEQ_CLS",
        modules_to_save=["classifier"]
    )
    return config
    

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",        
    bnb_4bit_use_double_quant=True,  
    #bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=["classifier"]
    #llm_int8_skip_modules=["classifier", "pre_classifier"]
)


In [11]:
logs = []
def training_pipeline_qlora(model_id, config, bnb_config, run_name,seed, train_ds, val_ds, test_ds):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        num_labels=2,
         use_safetensors=True
    )
    set_seed(seed)

    print(f"--- Training Run: {run_name} ---")
    base_model = prepare_model_for_kbit_training(base_model)
    model = get_peft_model(base_model, config)

    model.print_trainable_parameters()
    
    batch_size = 16
    learning_rate = 2e-5

    out_dir = f"./results/{model_id}_{run_name}_seed{seed}"
    args = TrainingArguments(
        output_dir=out_dir,
        remove_unused_columns=False,
        optim="paged_adamw_32bit",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=batch_size,
        bf16=True,        
        max_grad_norm=1.0,
        num_train_epochs=2,
        logging_steps=10,
        load_best_model_at_end=True,
        label_names=["labels"],

    )

    trainer = Trainer(
        model,
        args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
    )


    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    os.system("nvidia-smi")
    start = time.time()
    trainer.train()
    os.system("nvidia-smi")
    end = time.time()
    train_time = end - start
    if torch.cuda.is_available():
        peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        peak_vram = 0
    predictions = trainer.predict(test_ds)

    preds = np.argmax(predictions.predictions, axis=1)
    
    labels = predictions.label_ids
    
    acc = accuracy_score(labels, preds)
        
    f1 = f1_score(labels, preds, average="macro")

    print(f"[{run_name}] Accuracy: {acc:.4f}")
    print(f"[{run_name}] F1 Score: {f1:.4f}")
    print(f"[{run_name}] Training Time: {train_time:.2f} seconds\n")

    logs.append({
        "seed": seed,
        "rank": config.r,
        "lr": learning_rate,
        "run_name": run_name,
        "target": str(config.target_modules),
        "accuracy": acc,
        "f1": f1,
        "train_time": train_time,
        "vram": peak_vram
    })


    del model, trainer, base_model
    torch.cuda.empty_cache()


In [12]:
model_ids = ["FacebookAI/roberta-base"]
# "microsoft/deberta-v3-base",

for model_id in model_ids:
    configs_to_test = {
        f"qlora_r16_{model_id}": setup_config(16, get_target_modules(model_id)),
    }
    train_ds, val_ds, test_ds = prepare_datasets(model_id, ds)
    for run_name, config in configs_to_test.items():
        for seed in [42, 43]:
            training_pipeline_qlora(model_id, config, bnb_config, run_name, seed, train_ds, val_ds, test_ds)



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Training Run: qlora_r16_FacebookAI/roberta-base ---
trainable params: 1,181,954 || all params: 125,829,124 || trainable%: 0.9393


c:\Users\FAST.LAB12-PC29\anaconda3\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.342435,0.334639
2,1.070690,0.288377


...

--- Training Run: qlora_r16_FacebookAI/roberta-base ---
trainable params: 1,181,954 || all params: 125,829,124 || trainable%: 0.9393


c:\Users\FAST.LAB12-PC29\anaconda3\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.221053,0.316990
2,1.032944,0.283733


c:\Users\FAST.LAB12-PC29\anaconda3\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


[qlora_r16_FacebookAI/roberta-base] Accuracy: 0.8963
[qlora_r16_FacebookAI/roberta-base] F1 Score: 0.8963
[qlora_r16_FacebookAI/roberta-base] Training Time: 718.83 seconds



In [13]:
df = pd.DataFrame(logs)
df.to_csv("exp07_qlora_experiment_results.csv", index=False)